# TOPOTEX Technical Report — object-level baseline

> 全部 shape/参数量/SHA 由代码现场读取，Markdown 不手写数字。

**唯一正式主线**

```mermaid
flowchart TD
    A["Reference Image"] --> C["UniTEX Stage-1 six views (offline)"]
    B["Textureless Mesh"] --> C
    B --> D["Surface Conditioner<br/>(FaceTokenizer + MV CrossAttn + Topology Transformer)"]
    C --> D
    D --> E["Z_F [F,384]"]
    E --> Q["Factorized Dense UV Query Encoder<br/>(face addr 384→96 · bary MLP 27→96→96 · LN · Conv2d 8×8)"]
    Q --> F["Global UV Query Attention (1024 tokens, K=V=Z_F)"]
    F --> G["Flow Matching (Euler-50)"]
    G --> H["Texture [3,256,256]"]
    classDef data fill:#e8f0fe,stroke:#3366cc;
    classDef train fill:#fff3e0,stroke:#e8821a;
    class A,B,E,H data;
    class C,D,Q,F,G train;
```

**协议三句话**：split **by object**（GLB 资产为单位，500 unseen test）；
augment **by UV query**（native/xatlas/blender_smart 均匀 0.8 + partial 0.2）；
evaluate **on unseen objects**（不再以 same-object held-out UV 为主泛化轴）。

**分辨率区分**：source reference 512² / MV 256²（UniTEX 原生 512）/
native GT 256² / **UV query = model = 256²**。未来更高分辨率必须从
uv_vertices/uv_faces + 原始纹理重光栅化重烘焙，禁止 resize。

In [ ]:
import hashlib, json, os, sys, types
from pathlib import Path
p = Path.cwd()
PROJECT_ROOT = next(c for c in (p, *p.parents) if (c / "configs").exists())
sys.path.insert(0, str(PROJECT_ROOT))
DATA = Path(os.environ.get("TOPOTEX_DATA", "/root/youjiaZhang/topotex_data"))
RUN_ROOT = Path(os.environ.get("TOPOTEX_RUN_ROOT", DATA / "runs"))
RUN = Path(os.environ.get("REPORT_RUN", RUN_ROOT / "fm_accept_overfit1"))
DATASET_ROOT = Path(os.environ.get("TOPOTEX_DATASET_ROOT", DATA / "dataset"))
RANDOM_SEED = 20260727

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK SC",
                                          "Noto Sans CJK JP", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
from topotex import TopoTexDataset, TopoTexPipeline

split = json.loads((DATA / "object_split.json").read_text())
print("dataset  :", len(split["train"]) + len(split["val"]), "objects |",
      len(split["train"]), "train /", len(split["val"]), "unseen test")
for f in ("dataset_manifest.jsonl", "object_split.json"):
    print(f"{f:24s} sha256 {hashlib.sha256((DATA / f).read_bytes()).hexdigest()[:16]}…")
ck_file = next((RUN / n for n in ("ckpt.pt", "ckpt_final.pt")
                if (RUN / n).exists()), RUN)
pipe = TopoTexPipeline.from_checkpoint(ck_file, "cuda:0")
ck, model = pipe.checkpoint, pipe.model
cond, dit = model.conditioner, model.dit
n = lambda m: sum(x.numel() for x in m.parameters())
print(f"weights  : {RUN.name} @ step {ck.get('global_step')}")
print(f"params   : conditioner {n(cond)/1e6:.2f}M (decoder {n(cond.decoder)/1e6:.2f}M, Dq={cond.decoder.texel_dim})"
      f" + FM net {n(dit)/1e6:.2f}M = {(n(cond)+n(dit))/1e6:.2f}M")
print(f"encoder  : {ck['config'].get('uv_query_encoder')} | blocks {len(cond.decoder.blocks)}"
      f" | split_sha {str(ck['config'].get('split_sha256'))[:16]}…")
sid = ck["samples"][0]
it = TopoTexDataset(DATASET_ROOT, [sid], device="cuda:0")[0]
print("object   :", sid, "| F =", len(it["mesh"]["faces"]))

## 一个真实 object 贯穿全链 — source → 表示 → 生成 → 回贴

In [ ]:
from topotex.data.mesh import (CANONICAL_VIEWS, camera_matrices, dilate_texture,
                               linear_to_srgb_u8, rasterize_view,
                               render_albedo_rebake, seam_error)
V3 = it["mesh"]["vertices"].cpu().numpy().astype(np.float64)
F3 = it["mesh"]["faces"].cpu().numpy().astype(np.int64)
mvi = it["mv_images"]

def face_render(cols, vi, res=384):
    _, az, el = CANONICAL_VIEWS[vi]
    gb = rasterize_view(V3, F3, camera_matrices(az, el, V3.min(0), V3.max(0)), res)
    img = np.ones((res, res, 3)); img[gb["mask"]] = cols[gb["face_id"][gb["mask"]]]
    return img

def pca_rgb(x):
    x = np.asarray(x, np.float64); xc = x - x.mean(0)
    _, _, Vt = np.linalg.svd(xc[:: max(1, len(xc) // 4096)], full_matrices=False)
    pc = xc @ Vt[:3].T
    if pc.shape[1] < 3: pc = np.pad(pc, ((0, 0), (0, 3 - pc.shape[1])))
    lo, hi = np.percentile(pc, 2, 0), np.percentile(pc, 98, 0)
    return np.clip((pc - lo) / (hi - lo + 1e-9), 0, 1)

dec = cond.decoder
caps = {}
hooks = [dec.face_proj.register_forward_hook(lambda m, i, o: caps.__setitem__("addr", o.detach())),
         dec.patch_embed.register_forward_pre_hook(lambda m, i: caps.__setitem__("dense", i[0].detach()[0])),
         dec.patch_embed.register_forward_hook(lambda m, i, o: caps.__setitem__("tok", o.detach()[0].flatten(1).T))]
Z_F = pipe.encode(it["mesh"], mvi, it["graph"])
q0 = it["uv_queries"][0]
out = model.condition(Z_F, q0)
for h in hooks: h.remove()
vm0 = q0["valid_mask"].cpu().numpy()
tex = model.generate(out["uv_condition"], q0["valid_mask"], num_steps=50, seed=RANDOM_SEED)
im0 = ((tex.clamp(-1,1)+1)/2*255).round().byte().permute(1,2,0).cpu().numpy().copy(); im0[~vm0] = 0

def render_tex(q, t_u8, vi, res=384):
    canon = types.SimpleNamespace(vertices=V3, faces=F3)
    uvr = types.SimpleNamespace(uv_vertices=q["uv_vertices"].astype(np.float64),
                                uv_faces=q["uv_faces"], uv_face_to_mesh_face=np.arange(len(F3)))
    _, az, el = CANONICAL_VIEWS[vi]
    gb = rasterize_view(V3, F3, camera_matrices(az, el, V3.min(0), V3.max(0)), res)
    v = q["valid_mask"].cpu().numpy(); t = t_u8.copy(); t[~v] = 0
    return linear_to_srgb_u8(render_albedo_rebake(canon, uvr, dilate_texture(t, v), gb)), gb["mask"]

panels = [
    (np.asarray(Image.open(DATASET_ROOT / "samples" / sid / "reference.png").convert("RGB")), "reference 512²"),
    (mvi[0].permute(1, 2, 0).cpu().numpy(), "MV front 256²"),
    (face_render(np.full((len(F3), 3), 0.85), 0), "white mesh"),
    (face_render(pca_rgb(Z_F.cpu().numpy()), 0), "Z_F PCA"),
    (face_render(pca_rgb(caps["addr"].float().cpu().numpy()), 0), "face address PCA"),
    (pca_rgb(caps["tok"].float().cpu().numpy()).reshape(32, 32, 3), "UV tokens PCA"),
    (im0, "generated (native)"),
    (render_tex(q0, im0, 0)[0], "re-render"),
]
fig, axes = plt.subplots(2, 4, figsize=(15, 7.6))
for ax, (im, ttl) in zip(axes.ravel(), panels):
    ax.imshow(im, interpolation="nearest" if getattr(im, "shape", [0])[0] == 32 else None)
    ax.set_title(ttl, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

## 三个 full layouts + partial：同一 Z_F 的四次 query 与 seam

In [ ]:
ALLQ = {q["name"]: q for q in list(it["uv_queries"]) + list(it["test_uv_queries"])}
order = [("native", "uv_000"), ("xatlas", "uv_001"), ("blender_smart", "uv_test"), ("partial", "uv_002")]
def psnr(a, b, m):
    mse = float(((a[m]/255. - b[m]/255.)**2).mean()); return round(10*np.log10(1/max(mse,1e-12)), 2)
fig, axes = plt.subplots(2, 4, figsize=(14, 7.2))
mets = {}
preds = {}
for c_i, (nm, key) in enumerate(order):
    q = ALLQ[key]
    o = model.condition(Z_F, q)
    x = model.generate(o["uv_condition"], q["valid_mask"], num_steps=50, seed=RANDOM_SEED)
    im = ((x.clamp(-1,1)+1)/2*255).round().byte().permute(1,2,0).cpu().numpy().copy()
    vm = q["valid_mask"].cpu().numpy(); im[~vm] = 0
    preds[key] = im
    gt = (q["gt_texture"].permute(1,2,0).cpu().numpy()*255).astype(np.uint8)
    mets[nm] = psnr(gt, im, vm)
    axes[0][c_i].imshow(im); axes[0][c_i].set_title(f"{nm}  {mets[nm]:.1f} dB", fontsize=9)
    axes[1][c_i].imshow(gt)
for a in axes.ravel(): a.axis("off")
axes[1][0].set_ylabel("GT", fontsize=10)
plt.tight_layout(); plt.show()
q1 = ALLQ["uv_001"]; vm1 = q1["valid_mask"].cpu().numpy()
gt1 = (q1["gt_texture"].permute(1,2,0).cpu().numpy()*255).astype(np.uint8)
s_gen = seam_error(F3, q1["uv_vertices"], q1["uv_faces"], preds["uv_001"], vm1)
s_gt = seam_error(F3, q1["uv_vertices"], q1["uv_faces"], gt1, vm1)
print("UV PSNR:", mets)
print(f"seam generated {s_gen['seam_error']:.4f} vs GT floor {s_gt['seam_error']:.4f} (ratio {s_gen['seam_error']/max(s_gt['seam_error'],1e-9):.2f})")

## Baseline 训练与评估计划（数字由 split/profile 现场推导）
正式 run `fm_baseline_dim384_factorized`：随机初始化，8×H800、BF16、
packed K=4；评估在 500 unseen objects 上做 per-layout PSNR / render /
cross-layout consistency / seam（+ random32 / worst8 画廊）。

In [ ]:
prof = json.loads((RUN_ROOT / "fm_accept_smoke100" / "training_profile.json").read_text())
tail = [r for r in prof["rows"] if r["step"] >= 200]
mps = float(np.mean([r["meshes_per_sec"] for r in tail]))
n_train = len(split["train"])
steps = -(-n_train * 2000 // 32)
print(f"train objects {n_train} | global meshes/step 32 | total steps {steps:,}")
print(f"measured smoke100 throughput: {mps:.1f} mesh exposures/s "
      f"(median util {np.median([r['gpu_util'] for r in tail]):.0f}%)")
print(f"projected wall time: {steps*32/mps/3600:.1f} h")
ev = RUN_ROOT / "fm_baseline_dim384_factorized" / "eval.json"
if ev.exists():
    agg = json.loads(ev.read_text())["aggregate"]
    print("baseline eval:", {k: agg[k] for k in sorted(agg)[:8]})
else:
    print("baseline eval.json: pending (evaluation runs after full training)")